In [8]:
import torch.nn as nn
import torch

In [2]:
nnse = NNSELoss()
a = torch.randn(2, 500, 1)
b = torch.randn(2, 500, 1)
print(nnse(a, b))

In [7]:
# Example usage
batch_size = 4
num = 5
logits = torch.randn(batch_size, num, 3)  # Model output logits
targets = torch.randint(0, 3, (batch_size, num))  # Ground truth labels

# Initialize the custom loss
loss_fn = CustomCrossEntropyLoss()

# Compute the loss
loss = loss_fn(logits, targets)
print(loss)

tensor(1.2100)


In [ ]:
class TSSLoss(nn.Module):
    def __init__(self, threshold, lambda_weight=4.0, alpha=1.0, gamma=10.0, smooth_type="sigmoid"):
        super(TSSLoss, self).__init__()
        self.threshold = threshold
        self.lambda_weight = lambda_weight
        self.alpha = alpha
        self.gamma = gamma
        self.smooth_type = smooth_type
        self.mse = nn.MSELoss()
    
    def forward(self, y_pred, y_true):
        mse_loss = self.mse(y_pred, y_true)
        same_side = ((y_pred - self.threshold) * (y_true - self.threshold)) >= 0
        diff_side = ~same_side
        penalty = torch.zeros_like(y_pred)
        penalty[diff_side] = self._smooth_function(torch.abs(y_pred[diff_side] - self.threshold))
        cross_boundary_penalty = penalty * self.alpha * torch.abs(y_pred - self.threshold)
        classification_loss = cross_boundary_penalty.mean()
        total_loss = mse_loss + self.lambda_weight * classification_loss
        return total_loss
    
    def _smooth_function(self, x):
        if self.smooth_type == "sigmoid":
            return torch.sigmoid(self.gamma * x)
        elif self.smooth_type == "tanh":
            return (torch.tanh(self.gamma * x) + 1) / 2
        else:
            raise ValueError("Unsupported smooth type. Choose 'sigmoid' or 'tanh'.")